In [1]:
import pandas as pd
import numpy as np
import json
from glob import glob


In [2]:
files = ['/content/USvideos.csv',
         '/content/INvideos.csv',
         '/content/GBvideos.csv']


In [3]:
def clean_country(file_path, country_code):
    df = pd.read_csv(file_path, encoding='latin-1')

    # Fix columns
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]

    # Add country column
    df['country'] = country_code

    # Fix date formats
    df['publish_time'] = pd.to_datetime(df['publish_time'], errors='coerce')
    df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')

    # Convert numbers
    numeric_cols = ['views','likes','dislikes','comment_count']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # Remove duplicates
    df = df.drop_duplicates(subset=['video_id','trending_date','country'])

    # Simple feature engineering
    df['title_length'] = df['title'].astype(str).str.len()
    df['tag_count'] = df['tags'].astype(str).apply(lambda x: len(x.split('|')))

    return df


In [4]:
us = clean_country('/content/USvideos.csv', 'US')
ind = clean_country('/content/INvideos.csv', 'IN')
gb = clean_country('/content/GBvideos.csv', 'GB')


/tmp/ipython-input-1013757587.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')
/tmp/ipython-input-1013757587.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')
/tmp/ipython-input-1013757587.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')


In [5]:
all_data = pd.concat([us, ind, gb], ignore_index=True)

In [6]:
def load_category_map(json_file):
    with open(json_file) as f:
        data = json.load(f)
    items = data['items']
    mapping = {}
    for item in items:
        mapping[int(item['id'])] = item['snippet']['title']
    return mapping

us_map = load_category_map('/content/US_category_id.json')
in_map = load_category_map('/content/IN_category_id.json')
gb_map = load_category_map('/content/GB_category_id.json')


In [7]:
def map_category(df, mapping):
    df['category_name'] = df['category_id'].map(mapping)
    return df


In [8]:
us = map_category(us, us_map)
ind = map_category(ind, in_map)
gb = map_category(gb, gb_map)


In [9]:
all_data = pd.concat([us, ind, gb], ignore_index=True)


In [11]:
import os
os.makedirs('data/cleaned', exist_ok=True)
all_data.to_csv('data/cleaned/cleaned_youtube.csv', index=False)

In [12]:
all_data.head()
all_data.isnull().sum()
all_data.country.value_counts()


,count
country,
IN,22789
US,17725
GB,17440
